In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import MinMaxScaler #Kode ini digunakan untuk memanggil library MinMaxScaler dari sklearn yang berfungsi melakukan normalisasi data.

In [21]:
df = pd.read_csv('/content/diabetes.csv')

print("Data Awal:")
print(df.head())


Data Awal:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [22]:
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values.reshape(-1,1)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nData Train :", X_train.shape)
print("Data Test  :", X_test.shape)



Data Train : (614, 8)
Data Test  : (154, 8)


In [24]:
scaler = MinMaxScaler(feature_range=(0, 1)) #digunakan untuk mengubah nilai data ke rentang 0 sampai 1.

X_train_scaled = scaler.fit_transform(X_train) #digunakan untuk menghitung nilai minimum dan maksimum data training lalu melakukan normalisasi.
X_test_scaled = scaler.transform(X_test)
X_all_scaled = scaler.transform(X) #digunakan untuk menormalisasi seluruh data.

print("\nNormalisasi selesai.")
print("Contoh X_train_scaled:")
print(X_train_scaled[:5])



Normalisasi selesai.
Contoh X_train_scaled:
[[0.11764706 0.42211055 0.         0.         0.         0.
  0.09649872 0.        ]
 [0.52941176 0.56281407 0.67213115 0.38095238 0.         0.42026826
  0.51409052 0.48333333]
 [0.05882353 0.69849246 0.37704918 0.3015873  0.09810875 0.42771982
  0.24594364 0.01666667]
 [0.         0.80904523 0.40983607 0.         0.         0.32637854
  0.07514944 0.73333333]
 [0.35294118 0.67336683 0.6557377  0.58730159 0.43735225 0.68852459
  0.06831768 0.41666667]]


In [25]:
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)

    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }

    return parameters

In [26]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(int)

In [27]:
def forward_propagation(X, parameters):

    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]

    Z1 = np.dot(W1, X.T) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }

    return A2, cache


In [28]:
def compute_cost(A2, y):

    m = y.shape[0]

    cost = -1/m * np.sum(
        y.T * np.log(A2 + 1e-8) +
        (1 - y.T) * np.log(1 - A2 + 1e-8)
    )

    return cost

In [29]:
def backward_propagation(X, y, parameters, cache):

    m = X.shape[0]

    W2 = parameters["W2"]

    A1 = cache["A1"]
    A2 = cache["A2"]
    Z1 = cache["Z1"]

    dZ2 = A2 - y.T
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * relu_derivative(Z1)

    dW1 = (1/m) * np.dot(dZ1, X)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }

    return grads


In [30]:
def update_parameters(parameters, grads, learning_rate):

    parameters["W1"] = parameters["W1"] - learning_rate * grads["dW1"]
    parameters["b1"] = parameters["b1"] - learning_rate * grads["db1"]

    parameters["W2"] = parameters["W2"] - learning_rate * grads["dW2"]
    parameters["b2"] = parameters["b2"] - learning_rate * grads["db2"]

    return parameters

In [31]:
def train_ann(
    X_train,
    y_train,
    hidden_size=8,
    epochs=5000,
    learning_rate=0.01
):

    input_size = X_train.shape[1]
    output_size = 1

    parameters = initialize_parameters(
        input_size,
        hidden_size,
        output_size
    )
    for i in range(epochs):

        # FORWARD
        A2, cache = forward_propagation(X_train, parameters)

        # COST
        cost = compute_cost(A2, y_train)

        # BACKWARD
        grads = backward_propagation(
            X_train,
            y_train,
            parameters,
            cache
        )
         # UPDATE
        parameters = update_parameters(
            parameters,
            grads,
            learning_rate
        )

        # PRINT COST
        if i % 500 == 0:
            print(f"Epoch {i} | Cost : {cost:.6f}")

    return parameters


In [32]:
parameters = train_ann(
    X_train,
    y_train,
    hidden_size=8,
    epochs=5000,
    learning_rate=0.01
)

Epoch 0 | Cost : 0.693874
Epoch 500 | Cost : 0.591895
Epoch 1000 | Cost : 0.571215
Epoch 1500 | Cost : 0.557964
Epoch 2000 | Cost : 0.550360
Epoch 2500 | Cost : 0.542098
Epoch 3000 | Cost : 0.539070
Epoch 3500 | Cost : 0.535136
Epoch 4000 | Cost : 0.531883
Epoch 4500 | Cost : 0.528533


In [33]:
def predict(X, parameters):

    A2, _ = forward_propagation(X, parameters)

    predictions = (A2 > 0.5).astype(int)

    return predictions.T

In [34]:
all_predictions = predict(X, parameters)

hasil_prediksi = df.copy()
hasil_prediksi['Prediksi'] = all_predictions
hasil_prediksi['Keterangan Prediksi'] = hasil_prediksi['Prediksi'].map({
    0: 'Tidak Diabetes',
    1: 'Diabetes'
})
hasil_prediksi['Keterangan Asli'] = hasil_prediksi['Outcome'].map({
    0: 'Tidak Diabetes',
    1: 'Diabetes'
})

print("\n==============================")
print("HASIL PREDIKSI SELURUH DATA")
print("==============================")
print(hasil_prediksi)


HASIL PREDIKSI SELURUH DATA
     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0              6      148             72             35        0  33.6   
1              1       85             66             29        0  26.6   
2              8      183             64              0        0  23.3   
3              1       89             66             23       94  28.1   
4              0      137             40             35      168  43.1   
..           ...      ...            ...            ...      ...   ...   
763           10      101             76             48      180  32.9   
764            2      122             70             27        0  36.8   
765            5      121             72             23      112  26.2   
766            1      126             60              0        0  30.1   
767            1       93             70             31        0  30.4   

     DiabetesPedigreeFunction  Age  Outcome  Prediksi Keterangan Prediksi  \
0    

In [35]:
accuracy = accuracy_score(y, all_predictions)

print("\n======================")
print("HASIL EVALUASI MODEL")
print("======================")

print("\nAccuracy :")
print(accuracy)

print("\nConfusion Matrix :")
print(confusion_matrix(y, all_predictions))

print("\nClassification Report :")
print(classification_report(y, all_predictions))


HASIL EVALUASI MODEL

Accuracy :
0.7408854166666666

Confusion Matrix :
[[392 108]
 [ 91 177]]

Classification Report :
              precision    recall  f1-score   support

           0       0.81      0.78      0.80       500
           1       0.62      0.66      0.64       268

    accuracy                           0.74       768
   macro avg       0.72      0.72      0.72       768
weighted avg       0.75      0.74      0.74       768



In [36]:

print("\n===================================")
print("PREDIKSI SELURUH DATA DIABETES")
print("===================================")

for i in range(len(X)):

    # Ambil data satu per satu
    sample_data = X[i].reshape(1, -1)

    # Prediksi
    prediction = predict(sample_data, parameters)

    print(f"\nData Ke-{i+1}")
    print("----------------------------")

    print("Pregnancies              :", df.iloc[i]['Pregnancies'])
    print("Glucose                  :", df.iloc[i]['Glucose'])
    print("BloodPressure            :", df.iloc[i]['BloodPressure'])
    print("SkinThickness            :", df.iloc[i]['SkinThickness'])
    print("Insulin                  :", df.iloc[i]['Insulin'])
    print("BMI                      :", df.iloc[i]['BMI'])
    print("DiabetesPedigreeFunction :", df.iloc[i]['DiabetesPedigreeFunction'])
    print("Age                      :", df.iloc[i]['Age'])

    print("\nTarget Asli :", df.iloc[i]['Outcome'])
    print("Hasil Prediksi :", prediction[0][0])

    if prediction[0][0] == 1:
        print("Keterangan Prediksi : Pasien Terindikasi Diabetes")
    else:
        print("Keterangan Prediksi : Pasien Tidak Terindikasi Diabetes")

Output streaming akan dipotong hingga 5000 baris terakhir.
Age                      : 36.0

Target Asli : 0.0
Hasil Prediksi : 0
Keterangan Prediksi : Pasien Tidak Terindikasi Diabetes

Data Ke-436
----------------------------
Pregnancies              : 0.0
Glucose                  : 141.0
BloodPressure            : 0.0
SkinThickness            : 0.0
Insulin                  : 0.0
BMI                      : 42.4
DiabetesPedigreeFunction : 0.205
Age                      : 29.0

Target Asli : 1.0
Hasil Prediksi : 1
Keterangan Prediksi : Pasien Terindikasi Diabetes

Data Ke-437
----------------------------
Pregnancies              : 12.0
Glucose                  : 140.0
BloodPressure            : 85.0
SkinThickness            : 33.0
Insulin                  : 0.0
BMI                      : 37.4
DiabetesPedigreeFunction : 0.244
Age                      : 41.0

Target Asli : 0.0
Hasil Prediksi : 1
Keterangan Prediksi : Pasien Terindikasi Diabetes

Data Ke-438
----------------------------
Pr

Pada kode tersebut dilakukan proses normalisasi menggunakan MinMaxScaler untuk mengubah seluruh nilai fitur ke rentang 0 sampai 1. Proses ini membuat setiap fitur memiliki skala yang seimbang sehingga model ANN dapat belajar dengan lebih stabil dan optimal. Dengan adanya normalisasi, proses training menjadi lebih cepat, penurunan error lebih baik, dan akurasi model meningkat dibandingkan tanpa normalisasi. Selain itu, model menjadi lebih mampu membedakan data pasien diabetes dan tidak diabetes dengan hasil prediksi yang lebih akurat.